# FingerBall Stage 2H — Manual-guided Hand Landmark Annotation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2H_Manual_Guided_Annotation.ipynb)

**目的：** リリース前後で背景やボールに重なって誤検出されたMediaPipe 21点を、元動画の座標系でドラッグ修正します。元データは保持し、修正点・遮蔽点・補間点の由来をCSVとJSONに保存します。

**主な操作：** フレームを選択 → 近くの点をドラッグ → 必要に応じて「遮蔽・不明」に設定 → キーフレームを保存 → ほかのフレームを編集。対象フレームを飛ばす・評価用フレームを確保することも可能です。

**重要な限界：** 修正するのは**2D投影座標だけ**です。MediaPipeの相対Zや3D関節角を「正しくなった」とは扱いません。遮蔽点を勝手に真値化しません。

## 0. セットアップ（Colab対応のMatplotlibウィジェット）

In [ ]:
%pip -q install "ipympl>=0.9.4" "ipywidgets>=8.0" "opencv-python-headless>=4.8" "pandas>=2" "matplotlib>=3.8"

In [ ]:
from google.colab import drive,output
drive.mount('/content/drive')
output.enable_custom_widget_manager()
%matplotlib widget
from pathlib import Path
import copy,json,datetime,zipfile
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display,clear_output
from matplotlib.lines import Line2D

## 1. 入力・出力の設定

元動画を使用してください。Stage 2Dの`release_zoom_175_210.mp4`は画像座標が変わっているため、このNotebookのアノテーションには使用できません。元動画の場所が見つからなければ`VIDEO_PATH`を明示指定します。

In [ ]:
PROJECT_PATH='/content/drive/MyDrive/母指回内トラッキング'
SOURCE_STAGE='stage2d'
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
CSV_PATH=''                       # 空ならStage 2Dの21点CSV
VIDEO_PATH=''                     # 空ならMyDrive/母指回内トラッキング配下から元動画を探索
FIRST_FRAME,LAST_FRAME=185,198
KEY_FRAMES=[187,189,190,191,193,196]
THROWING_HAND='Left'              # 左右は確認のうえ手動指定
CROP_SIZE=300                     # 元動画の画像ピクセル、表示の一辺
AUTO_SAVE=True
assert THROWING_HAND in ('Left','Right')
project=Path(PROJECT_PATH)
if not project.is_dir():raise FileNotFoundError(project)
if CSV_PATH:
    csv_path=Path(CSV_PATH)
else:
    base=project/f'FingerBall_v2_{SOURCE_STAGE}_results'
    csv_path=base/VIDEO_STEM/'hand_21_landmarks_long.csv'
    if not csv_path.is_file():
        hits=sorted(base.glob('*/hand_21_landmarks_long.csv'))
        if not hits:raise FileNotFoundError('CSV_PATHを指定してください。21点CSVがありません')
        csv_path=hits[0]
if not csv_path.is_file():raise FileNotFoundError(csv_path)
if VIDEO_PATH:
    video_path=Path(VIDEO_PATH)
else:
    candidates=[p for p in project.rglob('*') if p.is_file()
       and p.suffix.lower() in ('.mp4','.mov','.avi')
       and VIDEO_STEM in p.stem and 'zoom' not in p.stem.lower()
       and 'annotated' not in p.stem.lower()]
    candidates=sorted(candidates,key=lambda p:(p.name!=VIDEO_STEM+'.mp4',len(p.parts)))
    if not candidates:
        raise FileNotFoundError('元動画が見つかりません。VIDEO_PATHに元動画の絶対パスを指定してください')
    video_path=candidates[0]
if not video_path.is_file():raise FileNotFoundError(video_path)
if any(t in video_path.stem.lower() for t in ('zoom','annotated','comparison')):
    raise ValueError('座標の異なる加工動画は使用できません。元動画VIDEO_PATHを指定してください')
out_dir=project/'FingerBall_v2_stage2h_results'/csv_path.parent.name
out_dir.mkdir(parents=True,exist_ok=True)
state_file=out_dir/'manual_annotation_session.json'
print('CSV:',csv_path)
print('ORIGINAL video:',video_path)
print('Saving into:',out_dir)

## 2. 動画とLandmarksの読み込み・座標一致チェック

In [ ]:
source_df=pd.read_csv(csv_path)
required={'frame','landmark_id','x_px','y_px','z_relative','detected'}
if required-set(source_df.columns):
    raise ValueError('CSV必須列不足: '+str(required-set(source_df.columns)))
source_df['frame']=pd.to_numeric(source_df.frame,errors='raise').astype(int)
source_df['landmark_id']=pd.to_numeric(source_df.landmark_id,errors='raise').astype(int)
if source_df.duplicated(['frame','landmark_id']).any():
    raise ValueError('frame/landmark_idに重複があります')
for c in ['x_px','y_px','z_relative','detected']:
    source_df[c]=pd.to_numeric(source_df[c],errors='coerce')
cap=cv2.VideoCapture(str(video_path))
if not cap.isOpened():raise RuntimeError('元動画をOpenCVで開けません')
TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
if not (0<=FIRST_FRAME<=LAST_FRAME<TOTAL):raise ValueError('指定フレーム範囲が動画外です')
for norm,pix,extent in [('x_norm','x_px',W),('y_norm','y_px',H)]:
    if norm in source_df:
        good=source_df[[norm,pix]].dropna()
        good=good[good[norm].abs()>1e-4]
        if len(good)>10:
            est=float(np.median(good[pix]/good[norm]))
            if abs(est-extent)>3:
                cap.release()
                raise ValueError(f'CSVの画像寸法は約{est:.1f}ですが、指定動画は{extent}です。加工動画ではなく元動画を確認')
FRAME_IDS=list(range(FIRST_FRAME,LAST_FRAME+1))
def get_video_frame(f):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
    ok,im=cap.read()
    if not ok:raise RuntimeError(f'frame {f} の読み込み失敗')
    return cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
original={};edits={};statuses={}
for f in FRAME_IDS:
    group=source_df[(source_df.frame==f)&source_df.landmark_id.between(0,20)].sort_values('landmark_id')
    arr=np.full((21,2),np.nan,float)
    if len(group)==21 and group.landmark_id.tolist()==list(range(21)):
        p=group[['x_px','y_px']].to_numpy(float)
        valid=np.isfinite(p).all(axis=1)&group.detected.fillna(0).to_numpy().astype(bool)
        arr[valid]=p[valid]
    original[f]=arr.copy()
    edits[f]=arr.copy()
    statuses[f]=['ai' if np.isfinite(arr[i]).all() else 'missing' for i in range(21)]
if state_file.is_file():
    saved=json.loads(state_file.read_text(encoding='utf-8'))
    if (saved.get('video_filename')==video_path.name and
        saved.get('csv_path')==str(csv_path) and saved.get('resolution')==[W,H]):
        for key,item in saved.get('frames',{}).items():
            f=int(key)
            if f not in edits:continue
            p=np.array(item['coords'],dtype=object)
            edits[f]=np.array([[np.nan if v is None else float(v) for v in pair] for pair in p],float)
            statuses[f]=list(item['statuses'])
        print('前回の修正セッションを復元しました')
    else:print('既存の保存ファイルは別動画/別CSVです。上書きしません（保存先を変更してください）')
print(f'Video: {W}x{H} / {TOTAL} frames / {FPS:.3f}fps')
print('Frame range:',FRAME_IDS[0],FRAME_IDS[-1])